# ZAIC 2026 LifeMate — ASR beam search (chạy được trên **L4**, không train)

Bản cuối (public **52.10**) giải mã Whisper bằng **greedy**. Beam search vẫn tất định và thường giảm WER mà không cần train lại.
Notebook này: (1) đo WER trên val bằng `asr_v8_large_holdout` (chưa thấy clip val) với beam 1 / 3 / 5;
(2) nếu beam tốt hơn greedy, chạy public với hệ thống cuối (`ft_v11_all_adapter` + `asr_v8_large_all`) + beam tốt nhất.
Chi phí: L4 khoảng 1.5–2 giờ (~3–4 unit). Không cần A100.

Thắng khi: WER giảm ≥ 0.3 điểm **và** public > 52.10. Ghi lại thời gian ASR/mẫu — Docker cần max < 20 s/mẫu.


In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BEAMS      = [1, 3, 5]                     # 1 = greedy (phải ra lại WER 10.08%)
BATCH      = 32
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ASR_HOLD   = DRIVE_DIR + "/asr_v8_large_holdout"   # chưa thấy clip val -> đo WER hợp lệ
ASR_ALL    = DRIVE_DIR + "/asr_v8_large_all"       # ASR của bản cuối
ADAPTER    = DRIVE_DIR + "/ft_v11_all_adapter"     # LLM của bản cuối
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_beam"
LLM = "/content/ft_final/merged"
print("BEAMS:", BEAMS)


In [ ]:
# ==== KIỂM TRA ĐÚNG FILE (chạy đầu tiên) ====
NOTEBOOK = "beam"
from google.colab import drive; drive.mount("/content/drive")
import os, zipfile
_z = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
assert os.path.exists(_z), "Drive không có MyDrive/ZAIC/lifemate_baseline.zip"
_v = zipfile.ZipFile(_z).read("VERSION").decode().strip() if "VERSION" in zipfile.ZipFile(_z).namelist() else "(không có VERSION — zip cũ)"
assert _v == "2026-11-14-1400", f"lifemate_baseline.zip trên Drive là bản {_v}, cần bản 2026-11-14-1400 → ghi đè bằng bản mới nhất (zip của repo)"
_need = ["asr_v8_large_holdout/config.json", "asr_v8_large_all/config.json", "ft_v11_all_adapter/adapter_config.json"]
_miss = [p for p in _need if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not _miss, "Thiếu trên Drive: " + ", ".join(_miss)
print(f"OK: notebook {NOTEBOOK} | zip {_v} | đủ: {_need}")


In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/scripts/detcheck.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)
assert "--beams" in open(f"{WORK}/lifemate/asr.py").read(), "lifemate_baseline.zip chưa có tuỳ chọn --beams -> ghi đè bản mới nhất"

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. WER trên clip val với `asr_v8_large_holdout` (greedy phải ra ~10.08%)


In [ ]:
%cd {WORK}
wer, sec = {}, {}
for B in BEAMS:
    t0 = time.time()
    !python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_HOLD} --dtype {DTYPE} --beams {B} --out asr_val_b{B}.json 2>&1 | grep -E "^WER" | tee {RES}/wer_b{B}.txt
    sec[B] = time.time() - t0
    wer[B] = float(open(f"{RES}/wer_b{B}.txt").read().split("WER=")[1].split("%")[0])
    print(f"beam {B}: WER {wer[B]:.2f}%  ({sec[B]:.0f}s cho 1100 clip)")
BEST = min(BEAMS, key=lambda b: (round(wer[b], 2), b))
print("WER:", wer, "| tốt nhất: beam", BEST, "| giảm so với greedy:", round(wer[1] - wer[BEST], 2), "điểm")
json.dump({"wer": wer, "sec": sec, "best": BEST}, open(f"{RES}/wer_beams.json", "w"))


## 2. Public với hệ thống cuối + beam tốt nhất (mốc **52.10**) — chỉ chạy nếu beam giảm WER ≥ 0.3


In [ ]:
%cd {WORK}
if wer[1] - wer[BEST] < 0.3:
    print("Beam không giảm WER đủ (< 0.3 điểm) -> giữ greedy, KHÔNG cần nộp public. Dừng ở đây, Disconnect runtime.")
else:
    !python -m lifemate.merge --base {BASE_MODEL} --adapter {ADAPTER} --out {LLM} 2>&1 | grep -vi warn | tail -1
    t0 = time.time()
    !python -m lifemate.asr --root {DATA} --split public_test --model {ASR_ALL} --dtype {DTYPE} --beams {BEST} --out asr_public.json 2>&1 | tail -1
    asr_sec = time.time() - t0
    !python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/public 2>&1 | tail -1
    SUB = f"submission_public_final_beam{BEST}.jsonl"
    shutil.copy("runs/public/submission.jsonl", f"{RES}/{SUB}")
    shutil.copy("runs/public/submission.jsonl", os.path.join(DRIVE_DIR, SUB))
    print(f"ASR public beam {BEST}: {asr_sec:.0f}s cho 634 mẫu ({asr_sec / 634:.2f}s/mẫu, batch 16, {GPU_NAME})")
    print("NỘP FILE:", os.path.join(DRIVE_DIR, SUB))
    %cd {DATA}/public_test
    !python -m zaic_tools --split public_test --root .. --validate {RES}/{SUB} 2>&1 | tail -3 | tee {RES}/validate_public.txt


## 3. Đóng gói


In [ ]:
z = shutil.make_archive("/content/results_beam", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_beam.zip")); print("saved results_beam.zip trên Drive")
